## Data Analytics

### Chapter 1: Data Cleaning

Data Cleaning is a crucial step in the data analytics process, ensuring that the dataset is accurate, consistent, and ready for analysis. This chapter covers essential techniques for cleaning and preparing your data.

- **Imputation of Missing Values**: How to handle and fill in `NaN` values in datasets.

- **Unique Values**: Techniques to identify and extract distinct values from a column.

- **Duplicate Values**: Methods to detect and manage redundant entries in your data.

#### Generate Synthetic Data

In [114]:
import numpy as np
import pandas as pd

# Reproducible randomness
np.random.seed(42)

In [115]:
names = [
    "James", "Chris", "Adam", "Marian", "Peter",
    "Kevin", "Linda", "Emma", "Sophia", "Daniel",
    "Noah", "Olivia", "Lucas", "Grace", "Henry",
    "Liam", "Ava", "Mia", "Nathan", "Chloe"
]

majors = ["AI", "IT", "CSC", "SW"]

n = 20

df = pd.DataFrame({
    "Name": np.array(names),
    "Age": np.random.randint(18, 21, n),
    "Major": np.random.choice(majors, n),
    "GPA": np.round(np.random.uniform(2.0, 4.0, n), 2)
})

# Introduce 20% missing values into selected columns
for column in ["Age", "Major", "GPA"]:
    missing_idx = np.random.choice(df.index, int(0.2 * n), replace=False)
    df.loc[missing_idx, column] = np.nan

df

,Name,Age,Major,GPA
0,James,NaN,SW,2.91
1,Chris,18.0,SW,NaN
2,Adam,NaN,NaN,2.40
3,Marian,20.0,AI,NaN
4,Peter,NaN,SW,NaN
5,Kevin,18.0,IT,NaN
6,Linda,NaN,IT,3.22
7,Emma,19.0,AI,2.34
8,Sophia,20.0,SW,2.13
9,Daniel,20.0,NaN,3.90


In [116]:
# Get Column  Non-Null Count  Dtype as DataFrame
df_info = pd.DataFrame(df.dtypes).rename(columns={0: 'column_type'})
df_info['Non-Null Count'] = df.count()
df_info[['Non-Null Count', 'column_type']]

,Non-Null Count,column_type
Name,20,object
Age,16,float64
Major,16,object
GPA,16,float64


#### 1.1. Removing Missing Values 

The simplest way to handle missing values is to remove them from the dataset. This can be done using the `dropna()` function in pandas.

In [117]:
# Count the number of missing values
print(df.isnull().sum())

Name     0
Age      4
Major    4
GPA      4
dtype: int64


In [118]:
# Remove rows containing at least one missing value
df_drop = df.dropna()

In [119]:
# Get info
df_drop_info = pd.DataFrame(df_drop.dtypes).rename(columns={0: 'column_type'})
df_drop_info['Non-Null Count'] = df_drop.count()
df_drop_info[['Non-Null Count', 'column_type']]

,Non-Null Count,column_type
Name,10,object
Age,10,float64
Major,10,object
GPA,10,float64


#### 1.2. Imputing Missing Values

Common methods for imputing missing values include replacing them with the mean, median, or mode of the column. 

This can be done using the `fillna()` function in pandas.

In [120]:
df_imputed = df.copy()

# Fill numerical columns
df_imputed["Age"] = df_imputed["Age"].fillna(round(df_imputed["Age"].median()))
df_imputed["GPA"] = df_imputed["GPA"].fillna(df_imputed["GPA"].mean())

# Fill categorical column
df_imputed["Major"] = df_imputed["Major"].fillna(df_imputed["Major"].mode()[0])

In [121]:
# Filter the samples with missing values in the original data
df[df.isnull().any(axis=1)]

,Name,Age,Major,GPA
0,James,NaN,SW,2.91
1,Chris,18.0,SW,NaN
2,Adam,NaN,NaN,2.40
3,Marian,20.0,AI,NaN
4,Peter,NaN,SW,NaN
5,Kevin,18.0,IT,NaN
6,Linda,NaN,IT,3.22
9,Daniel,20.0,NaN,3.90
16,Ava,19.0,NaN,2.24
19,Chloe,19.0,NaN,3.82


In [122]:
# Get these samples index
missing_idx = df[df.isnull().any(axis=1)].index.tolist()

# Now return df_imputed by filtering out previously missing values from df
df_imputed[df_imputed.index.isin(missing_idx)].head(10)

,Name,Age,Major,GPA
0,James,19.0,SW,2.910000
1,Chris,18.0,SW,2.914375
2,Adam,19.0,SW,2.400000
3,Marian,20.0,AI,2.914375
4,Peter,19.0,SW,2.914375
5,Kevin,18.0,IT,2.914375
6,Linda,19.0,IT,3.220000
9,Daniel,20.0,SW,3.900000
16,Ava,19.0,SW,2.240000
19,Chloe,19.0,SW,3.820000


#### 1.3. Unique Values

Unique values in a dataset can be identified using the `unique()` function in pandas. This is useful for understanding the diversity of data in a column.

In [123]:
# Retrieve unique values in every column
for column in df.columns:
    unique_values = df[column].unique()
    print(f"{column}:")
    print(unique_values)
    print(f"Number of unique values: {len(unique_values)}\n")

Name:
['James' 'Chris' 'Adam' 'Marian' 'Peter' 'Kevin' 'Linda' 'Emma' 'Sophia'
 'Daniel' 'Noah' 'Olivia' 'Lucas' 'Grace' 'Henry' 'Liam' 'Ava' 'Mia'
 'Nathan' 'Chloe']
Number of unique values: 20

Age:
[nan 18. 20. 19.]
Number of unique values: 4

Major:
['SW' nan 'AI' 'IT' 'CSC']
Number of unique values: 5

GPA:
[2.91  nan 2.4  3.22 2.34 2.13 3.9  3.93 3.62 2.61 2.2  3.37 2.88 2.24
 2.99 2.07 3.82]
Number of unique values: 17



You can also use the `nunique()` function to count the number of unique values in a column.

In [124]:
# Return the count of unique values
print(df.nunique())

Name     20
Age       3
Major     4
GPA      16
dtype: int64


In [125]:
# Return the count of the major column
print(df["Major"].unique())
print(df["Major"].nunique())

['SW' nan 'AI' 'IT' 'CSC']
4


#### 1.4. Duplicate Values

Duplicate values can be identified using the `duplicated()` function in pandas. This function returns a boolean Series indicating whether each row is a duplicate of a previous row.

The `keep=` parameter can be set to 'first', 'last', or False to specify which duplicates to mark as True.

In [126]:
# Create a temp dataset to showcase how Pandas handles duplicate values
data = {
    "Name": ["John", "Mary", "John", "Sally", "Mary", "John", "John", "John"],
    "Age": [40, 30, 40, 50, 30, 40, 50, 40],
    "City": [
        "Bergen", "Oslo", "Stavanger", "Oslo",
        "Oslo", "Stavanger", "Stavanger", "Stavanger"
    ]
}

df_demo = pd.DataFrame(data)

# Marks every occurrence of a duplicated row
df_demo.duplicated(keep=False)

0    False
1     True
2     True
3    False
4     True
5     True
6    False
7     True
dtype: bool

In [127]:
# keeps the first occurrence and marks only the later duplicates
df_demo.duplicated(keep="first")

0    False
1    False
2    False
3    False
4     True
5     True
6    False
7     True
dtype: bool

In [128]:
# keeps the last occurrence and marks the earlier duplicates
df_demo.duplicated(keep="last")

0    False
1     True
2     True
3    False
4    False
5     True
6    False
7    False
dtype: bool

In [129]:
# create a smaller DataFrame containing only the Name and Major columns
# Add a new student name Kevin studying IT
df_imputed.loc[len(df_imputed)] = ["Kevin", 20, "IT", 3.5]
student_small = df_imputed[["Name", "Major"]]

# identify rows where both columns match
duplicates = student_small[student_small.duplicated(keep=False)]
duplicates

,Name,Major
5,Kevin,IT
20,Kevin,IT


You can drop duplicates using the `drop_duplicates()` function, which removes duplicate rows from the DataFrame.

In [130]:
# Removing Duplicate Rows
student_unique = student_small.drop_duplicates()
'''If we want to keep the last occurrence instead:
# student_unique = student_small.drop_duplicates(keep="last")
Or remove all duplicated rows completely:
student_unique = student_small.drop_duplicates(keep=False)
'''

student_unique

,Name,Major
0,James,SW
1,Chris,SW
2,Adam,SW
3,Marian,AI
4,Peter,SW
5,Kevin,IT
6,Linda,IT
7,Emma,AI
8,Sophia,SW
9,Daniel,SW
